# VESM: Getting Started (Quickstart & Inference)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntranoslab/vesm/blob/main/notebooks/VESM_Getting_Started.ipynb)

### Requirements
- Python 3.9+ (recommended), `torch`, `transformers`, `huggingface_hub`, `numpy`, `matplotlib`, `seaborn`
- GPU is optional but recommended for speed

## Table of contents
1. [Setup & Imports](#setup--imports)
2. [Load a VESM Model](#load-a-model)
3. [Run Inference & Get Scores](#vesm_inference)
4. [Visualize Results](#visualize-results)  
5. [Download Prediction Scores](#download-prediction-scores)
6. [VESM3 Inference](#vesm3-inference)

<a id="setup--imports"></a>

## 1. Setup & Imports

In [ ]:
import os
from google.colab import userdata

# Retrieve Hugging Face Token from Colab Secrets and set environment variable
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Successfully set HF_TOKEN from Colab secrets!")
except Exception as e:
    print("Warning: Could not retrieve HF_TOKEN from Colab secrets.")
    print("Please ensure you have added 'HF_TOKEN' as a secret in the left-hand panel.")

In [1]:
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, EsmForMaskedLM
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter("ignore", FutureWarning) # ignore future warnings from transformers package
warnings.simplefilter("ignore", UserWarning) # ignore user warnings

In [2]:
local_dir = 'vesm' # local directory to store models

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    device = torch.device('cuda:0')
else:
    device = 'cpu'

CUDA available: True
GPU name: Tesla T4


<a id="load-a-model"></a>

## 2. Load VESM model

In [3]:
esm_dict = {
    "VESM_3B": 'facebook/esm2_t36_3B_UR50D',
    "VESM_650M": 'facebook/esm2_t33_650M_UR50D',
    "VESM_150M": 'facebook/esm2_t30_150M_UR50D',
    "VESM_35M": 'facebook/esm2_t12_35M_UR50D',
    "VESM3": "esm3_sm_open_v1"
}

def load_vesm(model_name="VESM_3B", local_dir="vesm", device='cuda'):
    if model_name in esm_dict:
        ckt = esm_dict[model_name]
    else:
        print("Model not found")
        return None
    # download weights
    hf_hub_download(repo_id="ntranoslab/vesm", filename=f"{model_name}.pth", local_dir=local_dir)
    # load base model
    if model_name == "VESM3":
      from esm.models.esm3 import ESM3
      model = ESM3.from_pretrained(ckt, device=device).to(torch.float)
      tokenizer = model.tokenizers.sequence
    else:
      model = EsmForMaskedLM.from_pretrained(ckt).to(device)
      tokenizer = AutoTokenizer.from_pretrained(ckt)
    # load pretrained VESM
    model.load_state_dict(torch.load(f'{local_dir}/{model_name}.pth'), strict=False)
    return model, tokenizer

We first load the VESM_3B checkpoint from Hugging Face

In [4]:
model_name = 'VESM_3B'
model, tokenizer = load_vesm(model_name, local_dir=local_dir, device=device)
sequence_vocabs = tokenizer.get_vocab()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/588 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

# VESM Inference

The following function is to get log-likelihood ratio (LLR) scores for all possible single missense mutations of a given sequence.

In [5]:
def get_LLR(sequence, esm_model, device='cuda'):
  """
    @param sequence: str, input protein sequence
    @param esm_model: loaded VESM model
    @param device: 'cuda' or 'cpu'
    @return: pd.DataFrame, LLR scores for all missense mutations
  """
  tokens = tokenizer(sequence, return_tensors='pt')
  batch_tokens = tokens['input_ids']
  esm_model.eval()
  with torch.no_grad():
    logits =torch.log_softmax(esm_model(batch_tokens.to(device),)['logits'],dim=-1)[0,:,:].cpu()
  tok = batch_tokens[0,:].cpu()
  wt_norm = logits[np.arange(len(tok)), tok].unsqueeze(1)
  LLR = logits - wt_norm
  LLR=LLR[1:-1,4:24].numpy()
  AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
  LLR_ = pd.DataFrame(LLR,columns=tokenizer.all_tokens[4:24],index=list(sequence)).T.loc[AAorder]
  LLR_.columns = [j.split('.')[0]+' '+str(i+1) for i,j in enumerate(LLR_.columns)]
  return LLR_

In [6]:
def get_LLR_batched(sequences, esm_model, device='cuda'):
    """
    Computes LLR scores for a list of sequences in parallel using batching.
    @param sequences: list of str, input protein sequences
    @param esm_model: loaded VESM model
    @param device: 'cuda' or 'cpu'
    @return: list of pd.DataFrame, LLR scores for each sequence
    """
    # Tokenize with padding enabled
    tokens = tokenizer(sequences, padding=True, return_tensors='pt')
    batch_tokens = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)

    esm_model.eval()
    with torch.no_grad():
        outputs = esm_model(batch_tokens, attention_mask=attention_mask)
        # Compute log-softmax over amino acid vocabulary
        logits = torch.log_softmax(outputs.logits, dim=-1).cpu()

    results = []
    AAorder = ['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
    vocab_indices = [tokenizer.convert_tokens_to_ids(aa) for aa in AAorder]

    for idx, seq in enumerate(sequences):
        # Extract token sequence without padding
        seq_len = attention_mask[idx].sum().item()
        # Slice to ignore <cls> and <eos> / <pad>
        tok = tokens['input_ids'][idx, 1:seq_len-1]
        seq_logits = logits[idx, 1:seq_len-1, vocab_indices]

        # Compute wt normalization
        wt_indices = torch.tensor([vocab_indices.index(t.item()) if t.item() in vocab_indices else 0 for t in tok])
        wt_norm = seq_logits[np.arange(len(tok)), wt_indices].unsqueeze(1)

        LLR = (seq_logits - wt_norm).numpy()

        LLR_df = pd.DataFrame(LLR, columns=AAorder, index=list(seq)).T
        LLR_df.columns = [f"{aa} {pos+1}" for pos, aa in enumerate(LLR_df.columns)]
        results.append(LLR_df)

    return results

<a id="vesm_inference"></a>
## 3. Inference on a protein sequence

In [7]:
!pip install biopython ete3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 121.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.3 MB/s  0:00:00
  Created wheel for ete3: filename=ete3-3.1.3-py3-none-any.whl size=2273899 sha256=564d4f162972eada1f7236e82515472a070978d8a02b69dee78c674d809c7d61
  Stored in directory: /root/.cache/pip/wheels/88/cc/c1/0e0d50d8a29caccf5d482d883af1bc9d9af5b043ea108008c8
Successfully built ete3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [biopython]


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from Bio import SeqIO

fasta_path = '/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/joint.fa'

# Load and print the sequences
sequences = list(SeqIO.parse(fasta_path, 'fasta'))
print(f"Successfully loaded {len(sequences)} sequences from {fasta_path}.")
for record in sequences[:5]:
    print(f"ID: {record.id}")
    print(f"Sequence: {record.seq[:50]}...")
    print(f"Length: {len(record.seq)}\n")

Successfully loaded 934 sequences from /content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/joint.fa.
ID: hg38
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: panPan3
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: panTro6
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: gorGor6
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: ponAbe3
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550



In [10]:
anage=pd.read_csv("/content/drive/MyDrive/AnAge_11_1_27.tsv",sep="\t",index_col=0)
sp=pd.read_csv("/content/drive/MyDrive/470mammals_species_key_forOneDrive(470mammals_species_key).csv",index_col=0,encoding='latin1')
sp.index=sp.sp_id
sp['scientific_name'] = sp['scientific_name'].apply(lambda x: ' '.join(x.split()[:2]) if len(x.split()) > 2 else x)
import re

# Drop rows with missing scientific names first if any
sp_clean = sp.dropna(subset=['scientific_name']).copy()

def get_numerical_suffix(sp_id):
    # Extract all digits at the end of the sp_id string using regex
    match = re.search(r'\d+$', str(sp_id))
    return int(match.group()) if match else -1

# Add temporary helper columns for sorting
sp_clean['suffix_val'] = sp_clean['sp_id'].apply(get_numerical_suffix)

# Sort by scientific_name and then by suffix_val in descending order
# This places the row with the highest suffix at the top for each species
sp_clean = sp_clean.sort_values(by=['scientific_name', 'suffix_val'], ascending=[True, False])

# Deduplicate by keeping only the first occurrence (the highest suffix)
sp_deduped = sp_clean.drop_duplicates(subset=['scientific_name'], keep='first')

# Drop the helper column
sp_deduped = sp_deduped.drop(columns=['suffix_val'])

# Update the main 'sp' variable and re-index
sp = sp_deduped
sp.index = sp.sp_id

print(f"Deduplicated to {len(sp)} unique scientific names.")
display(sp.head(10))
anage['Species']=[x+" "+y for x,y in zip(anage.Genus,anage.Species)]

Deduplicated to 429 unique scientific names.


,sp_id,common_name,scientific_name,clade,mammal,ncbi_taxonid,has_duplicate_taxonid,GenBank_id,scaffold_n50_kb,BUSCO_analysis,notes_NCBI,Old_or_New_World_monkey
sp_id,,,,,,,,,,,,
HLaciJub2,HLaciJub2,cheetah,Acinonyx jubatus,Carnivora,placental,32536,no,GCA_003709585.1,48500.0,"C:98.9%[S:98.4%,D:0.5%],F:0.3%,M:0.8%,n:14502",suppressed,NaN
HLacoCah1,HLacoCah1,Egyptian spiny mouse,Acomys cahirinus,Glires,placental,10068,no,GCA_004027535.1,65.4,NaN,not reference,NaN
HLacoRus1,HLacoRus1,golden spiny mouse,Acomys russatus,Glires,placental,60746,no,GCA_903995435.1,66500.0,"C:98.6%[S:97.0%,D:1.6%],F:0.4%,M:0.9%,n:13798",reference,NaN
HLaeoCin1,HLaeoCin1,hoary bat,Aeorestes cinereus,Chiroptera,placental,257879,no,GCA_011751065.1,35100.0,NaN,reference,NaN
HLaepMel1,HLaepMel1,impala,Aepyceros melampus,Artiodactyla,placental,9897,no,GCA_006408695.1,344.5,NaN,reference,NaN
HLailMel2,HLailMel2,giant panda,Ailuropoda melanoleuca,Carnivora,placental,9646,no,GCA_002007445.2,129200.0,NaN,replaced; contaminated,NaN
HLailFul2,HLailFul2,lesser panda,Ailurus fulgens,Carnivora,placental,9649,no,GCA_002007465.1,3000.0,NaN,DNA Zoo; reference,NaN
HLalcAlc1,HLalcAlc1,Eurasian elk,Alces alces,Artiodactyla,placental,9852,no,GCA_007570765.1,4100.0,NaN,not reference,NaN
HLallBul1,HLallBul1,Gobi jerboa,Allactaga bullata,Glires,placental,1041416,no,GCA_004027895.1,36.3,NaN,reference,NaN


In [63]:
# !pip install legacy-cgi
from ete3 import Tree

tree_path = '/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/labelled.nwk'
tree = Tree(tree_path, format=1)

# Print a basic summary of the loaded tree
print("Tree loaded successfully!")
print(f"Total leaves (tips): {len(tree.get_leaves())}")
print(f"Total nodes (including internal): {len(tree.get_descendants()) + 1}")
print("\nFirst 10 leaf names:")
print([leaf.name for leaf in tree.get_leaves()[:10]])

Tree loaded successfully!
Total leaves (tips): 468
Total nodes (including internal): 934

First 10 leaf names:
['hg38', 'panPan3', 'panTro6', 'gorGor6', 'ponAbe3', 'HLhylMol2', 'HLnomLeu4', 'rheMac10', 'HLmacFus1', 'HLmacFas6']


In [64]:
tree.prune(set([x.name for x in tree])&set(sp[sp.scientific_name.isin(anage.Species)].index), preserve_branch_length=True)


In [65]:
# Map FASTA sequence IDs to their corresponding SeqRecord objects
seq_dict = {record.id: record for record in sequences}

# Map ete3 tree node names to their sequences
node_to_seq = {}
for node in tree.traverse():
    if node.name in seq_dict:
        # Clean sequence by removing gaps '-' and replacing 'X' / invalid chars
        raw_seq = str(seq_dict[node.name].seq)
        cleaned_seq = "".join([amino for amino in raw_seq if amino in "ACDEFGHIKLMNPQRSTVWY"])
        node_to_seq[node.name] = cleaned_seq
    else:
        # Warn if a node's sequence is missing from the FASTA file
        print(f"Warning: No sequence found for node '{node.name}'")

In [66]:
gc.collect()

37

In [60]:
import os
import pandas as pd

# Directory to store the CSV results
output_dir = '/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/node_llr_results'
os.makedirs(output_dir, exist_ok=True)

print("Calculating sequence site-wise log likelihoods using batched VESM...")
print(f"Results will be saved to and loaded from: {output_dir}")

node_log_lh = {}
node_names = list(node_to_seq.keys())

# Identify which nodes still need calculation
pending_names = []
pending_sequences = []

for name in node_names:
    csv_path = os.path.join(output_dir, f"{name}_llr.csv")
    if os.path.exists(csv_path):
        # Load existing result
        node_log_lh[name] = pd.read_csv(csv_path, index_col=0)
    else:
        pending_names.append(name)
        pending_sequences.append(str(node_to_seq[name]))

print(f"Total nodes: {len(node_names)}")
print(f"Already processed: {len(node_log_lh)}")
print(f"Pending processing: {len(pending_names)}")

# Run batched inference only for pending items
batch_size = 4  # Safer batch size for length ~2550 to prevent CUDA OOM

if len(pending_names) > 0:
    for i in range(0, len(pending_names), batch_size):
        batch_names = pending_names[i : i + batch_size]
        batch_seqs = pending_sequences[i : i + batch_size]

        print(f"Processing batch {i // batch_size + 1}/{(len(pending_names) - 1) // batch_size + 1}: {batch_names}")

        # Run batched inference
        batch_results = get_LLR_batched(batch_seqs, model, device=device)

        # Save results immediately and add to our dictionary
        for name, result_df in zip(batch_names, batch_results):
            csv_path = os.path.join(output_dir, f"{name}_llr.csv")
            result_df.to_csv(csv_path)
            node_log_lh[name] = result_df

print(f"Successfully loaded/calculated LLRs for {len(node_log_lh)} nodes.")

Calculating sequence site-wise log likelihoods using batched VESM...
Results will be saved to and loaded from: /content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/node_llr_results
Total nodes: 668
Already processed: 668
Pending processing: 0
Successfully loaded/calculated LLRs for 668 nodes.


In [62]:
len(set(seq_dict.keys())&set(node_log_lh.keys()))

668

In [61]:
tree.prune(list(set(sp.scientific_name)&set(anage.Species)), preserve_branch_length=True)

ValueError: Node names not found: ['Acomys russatus', 'Heterohyrax brucei', 'Ictidomys tridecemlineatus', 'Madoqua kirkii', 'Ammotragus lervia', 'Procavia capensis', 'Oreotragus oreotragus', 'Beatragus hunteri', 'Eptesicus fuscus', 'Rattus norvegicus', 'Peromyscus leucopus', 'Kobus ellipsiprymnus', 'Mellivora capensis', 'Tolypeutes matacus', 'Tachyglossus aculeatus', 'Sigmodon hispidus', 'Cercopithecus mona', 'Arctocephalus gazella', 'Tursiops aduncus', 'Carollia perspicillata', 'Lagenorhynchus obliquidens', 'Oryx dammah', 'Eulemur mongoz', 'Graphiurus murinus', 'Lontra canadensis', 'Semnopithecus entellus', 'Vicugna pacos', 'Connochaetes taurinus', 'Peponocephala electra', 'Syncerus caffer', 'Sciurus vulgaris', 'Redunca redunca', 'Elaphurus davidianus', 'Macaca mulatta', 'Rousettus aegyptiacus', 'Choloepus didactylus', 'Nycticebus coucang', 'Camelus dromedarius', 'Bos grunniens', 'Pteronura brasiliensis', 'Lycaon pictus', 'Phoca vitulina', 'Ursus thibetanus', 'Neotoma lepida', 'Panthe

In [59]:
sp['scientific_name'] = sp['scientific_name'].apply(lambda x: ' '.join(x.split()[:2]) if len(x.split()) > 2 else x)
display(sp[sp.scientific_name.str.split(" ").str.len() > 2])

,sp_id,common_name,scientific_name,clade,mammal,ncbi_taxonid,has_duplicate_taxonid,GenBank_id,scaffold_n50_kb,BUSCO_analysis,notes_NCBI,Old_or_New_World_monkey
sp_id,,,,,,,,,,,,


In [35]:
sp=sp[sp.scientific_name.isin(anage.Species)]
sp.index=sp.sp_id

In [26]:
from Bio import Phylo, SeqIO


def extract_branch_substitutions(fasta_path, tree_path):
    """
    Extract amino acid substitutions along every branch.

    Returns
    -------
    dict
        {(parent_name, descendant_name): ["A123:G", ...]}

    Positions are 1-based alignment coordinates.
    Gaps and ambiguous/nonstandard amino acids are skipped.
    The FASTA must contain aligned sequences for all tree nodes,
    including both ancestors and tips.
    """
    sequences = {
        record_id: str(record.seq).upper()
        for record_id, record in
        SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta")).items()
    }
    tree = Phylo.read(tree_path, "newick")
    nodes = list(tree.find_clades())

    # Require matching sequences rather than silently omitting branches.
    if any(node.name is None for node in nodes):
        raise ValueError("Use the FastML tree with labelled internal nodes.")

    missing = sorted({node.name for node in nodes} - sequences.keys())
    if missing:
        raise ValueError(f"Tree nodes missing from FASTA: {missing}")

    if len({len(sequences[node.name]) for node in nodes}) != 1:
        raise ValueError("All sequences must have the same aligned length.")

    amino_acids = set("ACDEFGHIKLMNPQRSTVWY")
    substitutions = {}

    for parent in nodes:
        for child in parent.clades:
            substitutions[(parent.name, child.name)] = [
                f"{parent_aa} {position}:{child_aa}"
                for position, (parent_aa, child_aa) in enumerate(
                    zip(sequences[parent.name], sequences[child.name]),
                    start=1,
                )
                if parent_aa != child_aa
                and parent_aa in amino_acids
                and child_aa in amino_acids
            ]

    return substitutions

In [59]:
len(set(seq_dict.keys())-set(node_log_lh.keys()))

266

In [70]:
branch_sum_LLR

{('N1', 'N2'): -72.3592098,
 ('N1', 'HLtacAcu1'): -19.6699448,
 ('N1', 'HLornAna3'): -27.3145913,
 ('N2', 'N3'): -208.7171273,
 ('N2', 'N449'): -125.056983,
 ('N3', 'N4'): 0.0,
 ('N3', 'N432'): -6.0746822,
 ('N4', 'N5'): -13.1791093,
 ('N4', 'N162'): -17.230697900000003,
 ('N5', 'N6'): 0.0,
 ('N5', 'N161'): -4.74544,
 ('N6', 'N7'): -10.2132457,
 ('N6', 'N68'): 0.0,
 ('N8', 'N9'): 0.0,
 ('N8', 'N51'): -11.677540200000001,
 ('N9', 'N10'): -12.5465501,
 ('N9', 'tarSyr2'): -31.0861283,
 ('N10', 'N11'): 0.0,
 ('N10', 'N39'): 0.0,
 ('N11', 'N12'): 0.0,
 ('N11', 'N18'): -6.9484525,
 ('N12', 'N13'): 0.0,
 ('N12', 'N17'): -1.8946754,
 ('N14', 'N15'): -3.385341,
 ('N14', 'gorGor6'): 0.0,
 ('N15', 'hg38'): 0.0,
 ('N15', 'N16'): 0.0,
 ('N16', 'panPan3'): -4.1355643,
 ('N16', 'panTro6'): 0.0,
 ('N17', 'HLhylMol2'): 0.0,
 ('N17', 'HLnomLeu4'): 0.0,
 ('N18', 'N19'): 0.0,
 ('N18', 'N32'): 0.0,
 ('N19', 'N20'): 0.0,
 ('N19', 'N28'): 0.0,
 ('N20', 'N21'): 0.0,
 ('N20', 'N24'): 0.0,
 ('N21', 'N22'): 0.0,

In [1]:
branch_sum_LLR={}
for pair in subs:

  try:
    aas=[x.split(":") for x in subs[pair]]

    dfi=node_log_lh[pair[0]]
    branch_sum_LLR[pair]=float(np.sum([dfi.loc[x[1], x[0]] for x in aas]))
  except:
    print(pair)


NameError: name 'subs' is not defined

In [49]:
'N8' in node_log_lh.keys()

True

In [27]:
subs = extract_branch_substitutions("/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/joint.fa", "/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/labelled.nwk")

for (parent, child), changes in subs.items():
    print(f"{parent} -> {child}: {', '.join(changes)}")

N1 -> N2: M 20:T, S 101:A, V 343:M, V 352:A, N 371:D, I 375:Q, N 572:S, A 1091:T, T 1205:N, T 1239:S, E 1243:D, T 1244:A, Q 1594:H, N 1785:S, I 1861:V, L 1879:M
N1 -> HLtacAcu1: T 317:A, S 577:G, S 1171:P, A 2467:S
N1 -> HLornAna3: S 12:P, T 14:V, L 350:I, V 1043:I, P 1088:T, A 1856:G
N2 -> N3: R 2:L, A 4:T, S 5:G, T 6:P, S 12:A, A 13:T, M 15:T, G 17:S, T 18:N, T 20:S, I 21:V, S 33:N, S 211:A, Y 238:F, A 338:S, I 342:L, L 350:S, I 476:M, S 569:G, S 572:T, V 746:I, R 1024:K, S 1046:T, S 1086:N, T 1091:I, N 1098:A, D 1126:E, A 1127:V, A 1130:P, A 1131:S, L 1165:Q, S 1240:G, S 1409:T, R 1635:K, N 1645:S, H 1703:N, A 1734:T, S 1779:A, S 1824:N, N 1843:S, A 1856:T, V 1861:T, V 1865:L, M 2457:I, S 2460:G, T 2466:P
N2 -> N449: R 2:P, A 4:P, M 50:L, G 98:A, F 140:C, N 193:T, A 347:T, R 403:Q, D 457:E, H 470:P, V 1081:L, V 1211:M, L 1237:M, S 1240:N, T 1508:I, M 1724:A, H 1739:R, Q 1741:H, I 1825:V, N 1843:H, E 1857:D, I 2150:V, N 2439:T, M 2457:L
N3 -> N4: 
N3 -> N432: A 1828:T, A 1842:T
N4 ->

compute log-likelihood ratio scores (LLR) for all missense mutations with VESM

In [ ]:
get_LLR(protein_sequence, model)

<a id="visualize-results"></a>

## 4. Visualizing Results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
from google.colab import output

AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']

def plot_interactive(LLR,higher_than_wt=False,thresh=2,zmax=0,cmap='Viridis_r'):

  TITLE=''

  is_dark = output.eval_js('document.documentElement.matches("[theme=dark]")')
  template='plotly_dark' if is_dark else 'plotly_white'

  fig = px.imshow(LLR.values, x=LLR.columns, y=LLR.index, color_continuous_scale=cmap,zmax=zmax,
                  labels=dict(y="Amino acid change", x="Protein sequence", color="LLR"),
                  template=template,
                  title=TITLE)
  fig.update_xaxes(tickangle=-90,range=[0,99],rangeslider=dict(visible=True),dtick=1)
  fig.update_yaxes(dtick=1)
  fig.update_layout({
  'plot_bgcolor': 'rgba(0, 0, 0, 0)',
  'paper_bgcolor': 'rgba(0, 0, 0, 0)',
  },font={'family':'Arial','size':11},
  hoverlabel=dict(font=dict(family='Arial', size=14)))

  fig.update_traces(
      hovertemplate="<br>".join([
          "<b>%{x} %{y}</b>"+
          " (%{z:.2f})",
      ])+'<extra></extra>'
  )
  if higher_than_wt:
    hwt_x=[]
    hwt_y=[]
    cust=[]
    for i in LLR.columns:
      for j in list(LLR.index[LLR[i]>thresh]):
        hwt_x+=[i]
        hwt_y+=[j]
        cust+=[LLR.loc[j,i]]

    fig.add_trace(go.Scatter(
        x=hwt_x,
        y=hwt_y,
        customdata=cust,
        mode='markers',
        marker=dict(size=8),
        hovertemplate="<br>".join([
            "<b>%{x} %{y}</b>"+
            " (%{customdata:.2f})",
        ])+'<extra></extra>')
    )

  fig.show()


def plot_heatmap(LLR,figname=None,vmin=None):
  """
    Plot a per-residue score heatmap.
      Args:
          scores: 2D array-like of shape (positions, alphabet) or 1D positional scores.
          figname: Optional basename to save PNG.
          range: Color scale range for heatmap.
  """
  primaryLLR=LLR
  plt.figure(figsize=(int(np.round(primaryLLR.shape[1]*90/390)),5))
  sns.heatmap( primaryLLR ,cmap='viridis_r',xticklabels=True, yticklabels=True,vmax=0,vmin=vmin)
  if not figname is None:
    plt.savefig(f"{figname}.png", dpi=300,bbox_inches = 'tight')
  plt.show()

### Visualizing LLRs

In [ ]:
# MC4R
sequence = 'MVNSTHRGMHTSLHLWNRSSYRLHSNASESLGKGYSDGGCYEQLFVSPEVFVTLGVISLLENILVIVAIAKNKNLHSPMYFFICSLAVADMLVSVSNGSETIVITLLNSTDTDAQSFTVNIDNVIDSVICSSLLASICSLLSIAVDRYFTIFYALQYHNIMTVKRVGIIISCIWAACTVSGILFIIYSDSSAVIICLITMFFTMLALMASLYVHMFLMARLHIKRIAVLPGTGAIRQGANMKGAITLTILIGVFVVCWAPFFLHLIFYISCPQNPYCVCFMSHFNLYLILIMCNSIIDPLIYALRSQELRKTFKEIICCYPLGGLCDLSSRY'
plot_interactive(get_LLR(sequence, model))

<a id="visualize_structure"></a>

### Visualize on Structure

In [ ]:
!pip install --quiet py3Dmol biopython

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
from Bio.SeqUtils import seq1
from Bio.PDB import PDBParser

def show_pdb(position_scores, pdb_file, width=800, height=600, surface=False):
    view = py3Dmol.view(width=width, height=height, js='https://3dmol.org/build/3Dmol.js')
    view.addModel(open(pdb_file, 'r').read(), 'pdb')
    sigmoid_scores = 1 / (1 + np.exp((0.6 * np.array(position_scores) + 6)))
    for i, value in enumerate(sigmoid_scores):
        rgba = plt.cm.coolwarm(value)
        hexcol = matplotlib.colors.rgb2hex(rgba[:3])
        view.setStyle({'resi': str(i+1)}, {'cartoon': {'color': hexcol}})
    view.setBackgroundColor('#383838')

    if surface:
        for i, value in enumerate(sigmoid_scores):
            rgba = plt.cm.coolwarm(value)
            hexcol = matplotlib.colors.rgb2hex(rgba[:3])
            view.addSurface(
                py3Dmol.VDW,
                { 'opacity': 0.7, 'color': hexcol },
                { 'resi': str(i+1) }
            )
        view.setBackgroundColor('white')


    view.zoomTo()
    return view

In [ ]:
import requests
def download_latest_af_pdb(uniprot_id, version_start=20, version_end=1, out_name=None):
    """
    Download the latest AF PDB model version for a UniProt ID.

    Args:
        uniprot_id (str): UniProt accession, e.g., "P12345"
        version_start (int): Highest version to check, e.g., 10
        version_end (int): Lowest version to check, e.g., 1
        out_name (str): Optional output filename. If None, use the server's name.
    """
    base_url = "https://alphafold.ebi.ac.uk/files"

    for v in range(version_start, version_end - 1, -1):
        url = f"{base_url}/AF-{uniprot_id}-F1-model_v{v}.pdb"
        head = requests.head(url)

        if head.status_code == 200:
            print(f"Found version v{v}: {url}")
            response = requests.get(url)
            response.raise_for_status()

            # Determine filename
            filename = out_name or url.split("/")[-1]
            with open(filename, "wb") as f:
                f.write(response.content)
            print(f"Downloaded to: {filename}")
            return filename

    print(f"No model found for UniProt ID '{uniprot_id}' in versions v{version_start}–v{version_end}.")
    return None

Example structure from AFDB

In [ ]:
uniprot_id = 'P32245'
pdb_file = download_latest_af_pdb(uniprot_id, version_start=10, version_end=4, out_name=None)

extract sequence from pdb

In [ ]:
sequence = "".join(
    seq1(res.get_resname())
    for res in PDBParser(QUIET=True).get_structure("model", pdb_file).get_residues()
    if res.id[0] == " "
)

average scores per position

In [ ]:
position_scores = get_LLR(sequence, model).values.mean(0)
show_pdb(position_scores,pdb_file)

<a id="download"></a>
## Download Prediction Scores

- **sequence**: input any amino acid sequence for LLR inference.
- **seq_name**: name the sequence for the file name (optional).
- **heatmap**: check if you want to download the LLR heatmap

In [ ]:
sequence = 'MVNSTHRGMHTSLHLWNRSSYRLHSNASESLGKGYSDGGCYEQLFVSPEVFVTLGVISLLENILVIVAIAKNKNLHSPMYFFICSLAVADMLVSVSNGSETIVITLLNSTDTDAQSFTVNIDNVIDSVICSSLLASICSLLSIAVDRYFTIFYALQYHNIMTVKRVGIIISCIWAACTVSGILFIIYSDSSAVIICLITMFFTMLALMASLYVHMFLMARLHIKRIAVLPGTGAIRQGANMKGAITLTILIGVFVVCWAPFFLHLIFYISCPQNPYCVCFMSHFNLYLILIMCNSIIDPLIYALRSQELRKTFKEIICCYPLGGLCDLSSRY' #@param {type:"string"}
seq_name = "" #@param {type:"string"}
heatmap = True #@param {type:"boolean"}

import hashlib, zipfile
from google.colab import files

def short_hash(seq, length=8):
    return hashlib.sha1(seq.encode()).hexdigest()[:length]

def meltLLR(LLR,gene_prefix=None,ignore_pos=False):
  vars = LLR.melt(ignore_index=False)
  vars['variant'] = [''.join(i.split(' '))+j for i,j in zip(vars['variable'],vars.index)]
  vars['score'] = vars['value']
  vars = vars.set_index('variant')
  if not ignore_pos:
    vars['pos'] = [int(i[1:-1]) for i in vars.index]
  del vars['variable'],vars['value']
  if gene_prefix is not None:
    vars.index=gene_prefix+'_'+vars.index
  return vars

base_name = f'vesm_LLR_{seq_name if len(seq_name) > 0 else short_hash(sequence)}'
csv_file = f"{base_name}.csv"
LLR_scores = get_LLR(sequence, model)

# save csv file
meltLLR(LLR_scores).to_csv(csv_file)

if heatmap:
    png_file = f"{base_name}.png"
    plot_heatmap(LLR_scores, figname=base_name, vmin=None)

    # Create results zip with both files
    zip_name = f"{base_name}_results.zip"
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        zipf.write(csv_file)
        zipf.write(png_file)

file_name = zip_name if heatmap else csv_file

print(f"Saved results to: {file_name}")
files.download(file_name)

<a id="VESM3"></a>

# VESM3 Inference

Downloading the base ESM3-open model requires huggingface login

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install --quiet esm

remove previous model (necessary if using Colab's T4 GPU)

In [ ]:
if 'model' in globals():
    del model
    torch.cuda.empty_cache()
    import gc; gc.collect()

Load VESM3 checkpoint

In [ ]:
vesm3, _ = load_vesm("VESM3", local_dir=local_dir, device=device)

In [ ]:
def get_vesm3_LLR_sequence(sequence, esm3_model):
  from esm.sdk.api import ESMProtein
  protein = ESMProtein(sequence=sequence)
  tokens = esm3_model.encode(protein).sequence
  with torch.no_grad():
    logits =torch.log_softmax(esm3_model.forward(sequence_tokens=tokens.reshape(1,-1)).sequence_logits[0, :, :], dim=-1).cpu()
  tok = tokens.cpu()
  wt_norm = logits[np.arange(len(tok)), tok].unsqueeze(1)
  LLR = logits - wt_norm

  AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
  order = [esm3_model.tokenizers.sequence.vocab[x] for x in AAorder]
  LLR=LLR[1:-1,order].numpy()
  LLR_ = pd.DataFrame(LLR,columns=AAorder,index=list(sequence)).T
  LLR_.columns = [j.split('.')[0]+' '+str(i+1) for i,j in enumerate(LLR_.columns)]
  return LLR_


def get_vesm3_LLR_structure(pdb_file, esm3_model):
  from esm.sdk.api import ESMProtein
  protein = ESMProtein.from_pdb(pdb_file)
  tokens = esm3_model.encode(protein)
  seq_tokens = tokens.sequence
  struct_tokens = tokens.structure

  with torch.no_grad():
    logits =torch.log_softmax(esm3_model.forward(sequence_tokens=seq_tokens.reshape(1,-1), structure_tokens=struct_tokens.reshape(1,-1) ).sequence_logits[0, :, :], dim=-1).cpu()
  tok = seq_tokens.cpu()
  wt_norm = logits[np.arange(len(tok)), tok].unsqueeze(1)
  LLR = logits - wt_norm

  AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
  order = [esm3_model.tokenizers.sequence.vocab[x] for x in AAorder]
  LLR=LLR[1:-1,order].numpy()
  LLR_ = pd.DataFrame(LLR,columns=AAorder,index=list(sequence)).T
  LLR_.columns = [j.split('.')[0]+' '+str(i+1) for i,j in enumerate(LLR_.columns)]
  return LLR_

In [ ]:
### Example structure from AFDB
uniprot_id = 'P32245'
pdb_file = download_latest_af_pdb(uniprot_id, version_start=10, version_end=1, out_name=None)

In [ ]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
# extract sequence from pdb
sequence = "".join(
    seq1(res.get_resname())
    for res in PDBParser(QUIET=True).get_structure("model", pdb_file).get_residues()
    if res.id[0] == " "
)

### Inference with sequence only

In [ ]:
LLR_seq = get_vesm3_LLR_sequence(sequence, vesm3)

### Inference with structure

In [ ]:
LLR_struct = get_vesm3_LLR_structure(pdb_file, vesm3)

### Visualize Results

Visualize differences between sequence- and structure- derived LLRs with VESM3

In [ ]:
# LLR_seq vs LLR_struct
import matplotlib
import matplotlib.pyplot as plt
plt.plot(LLR_seq.values.reshape(-1), LLR_struct.values.reshape(-1), '.')
plt.plot([-20, 5], [-20, 5], 'k--')
plt.xlabel('LLR from Sequence')
plt.ylabel('LLR from Structure')
plt.grid(True)
plt.show()

Plot the heatmap of differences

In [ ]:
def plot_diff_heatmap(diff,figname=None,range=4):
  plt.figure(figsize=(int(np.round(diff.shape[1]*90/390)),5))
  sns.heatmap( diff ,cmap='coolwarm_r',xticklabels=True, yticklabels=True,vmax=range,vmin=-range)
  if not figname is None:
    plt.savefig(f"{figname}.png", dpi=300,bbox_inches = 'tight')
  plt.show()

diff = LLR_struct - LLR_seq
plot_heatmap(LLR_seq)
threshold = -5
plot_diff_heatmap(diff[((LLR_struct<threshold) & (LLR_seq>threshold)) | ((LLR_struct>threshold) & (LLR_seq<threshold))])
plot_heatmap(LLR_struct)